# 🔐 SigAuth Complete Backend - Production Ready

## Advanced Forgery-Resilient Signature Authentication
### Using Siamese Metric Learning and Digital Tamper Detection

---

**Features:**
- Siamese Network with ResNet-18 backbone for skilled forgery detection
- CNN-based Tamper Detection for digital copy-paste forgeries
- Complete training pipeline with metrics (FAR, FRR, EER, Precision, Recall, F1)
- Flask API for frontend integration via ngrok

**Datasets Required (upload as ZIP files):**
1. `task1.zip` - SVC2004 Task 1 signature images
2. `task2.zip` - SVC2004 Task 2 signature images  
3. `synthetic_tamper.zip` - Clean and tampered signature images

## 📦 Cell 1: Install Dependencies

In [6]:
# Install required packages
!pip install flask flask-cors pyngrok pillow scikit-learn matplotlib seaborn tqdm -q

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models

import os
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, roc_curve, auc
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Using device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

✅ Using device: cpu


## 📁 Cell 2: Upload and Extract Datasets

**IMPORTANT:** Upload your ZIP files using the file browser on the left panel:
1. Click the folder icon 📁
2. Click the upload button ⬆️
3. Upload: `task1.zip`, `task2.zip`, `synthetic_tamper.zip`
4. Run this cell AFTER uploading

In [7]:
import os
import zipfile

# Create directories
os.makedirs('datasets/svc2004/task1', exist_ok=True)
os.makedirs('datasets/svc2004/task2', exist_ok=True)
os.makedirs('datasets/tamper/clean', exist_ok=True)
os.makedirs('datasets/tamper/tampered', exist_ok=True)
os.makedirs('saved_models', exist_ok=True)
os.makedirs('results', exist_ok=True)

def extract_zip(zip_path, extract_to):
    """Extract ZIP file to specified directory"""
    if os.path.exists(zip_path):
        print(f"📦 Extracting {zip_path}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)
        print(f"   ✅ Extracted to {extract_to}")
        return True
    else:
        print(f"   ⚠️ {zip_path} not found. Please upload it.")
        return False

# Extract datasets
print("="*50)
print("📂 EXTRACTING DATASETS")
print("="*50)

# Try different possible paths for uploaded files
task1_paths = ['task1.zip', '/content/task1.zip', 'Task1.zip', '/content/Task1.zip']
task2_paths = ['task2.zip', '/content/task2.zip', 'Task2.zip', '/content/Task2.zip']
tamper_paths = ['synthetic_tamper.zip', '/content/synthetic_tamper.zip', 'tamper.zip', '/content/tamper.zip']

for path in task1_paths:
    if extract_zip(path, 'datasets/svc2004/task1'):
        break

for path in task2_paths:
    if extract_zip(path, 'datasets/svc2004/task2'):
        break

for path in tamper_paths:
    if extract_zip(path, 'datasets/tamper'):
        break

print("\n" + "="*50)
print("📊 DATASET SUMMARY")
print("="*50)

def count_images(folder):
    """Recursively count image files"""
    count = 0
    if os.path.exists(folder):
        for root, dirs, files in os.walk(folder):
            for f in files:
                if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
                    count += 1
    return count

task1_count = count_images('datasets/svc2004/task1')
task2_count = count_images('datasets/svc2004/task2')
clean_count = count_images('datasets/tamper/clean')
tampered_count = count_images('datasets/tamper/tampered')

print(f"Task 1 images: {task1_count}")
print(f"Task 2 images: {task2_count}")
print(f"Clean images: {clean_count}")
print(f"Tampered images: {tampered_count}")
print(f"\nTotal: {task1_count + task2_count + clean_count + tampered_count} images")

📂 EXTRACTING DATASETS
   ⚠️ task1.zip not found. Please upload it.
   ⚠️ /content/task1.zip not found. Please upload it.
📦 Extracting Task1.zip...
   ✅ Extracted to datasets/svc2004/task1
   ⚠️ task2.zip not found. Please upload it.
   ⚠️ /content/task2.zip not found. Please upload it.
📦 Extracting Task2.zip...
   ✅ Extracted to datasets/svc2004/task2
📦 Extracting synthetic_tamper.zip...
   ✅ Extracted to datasets/tamper

📊 DATASET SUMMARY
Task 1 images: 1600
Task 2 images: 1600
Clean images: 0
Tampered images: 1550

Total: 4750 images


## 🧠 Cell 3: Define Model Architectures

In [8]:
class SiameseNetwork(nn.Module):
    """
    Siamese Network with ResNet-18 backbone for signature verification.
    Uses metric learning to compare signature embeddings.
    """
    def __init__(self, embedding_dim=128):
        super(SiameseNetwork, self).__init__()

        # ResNet-18 backbone (pretrained)
        resnet = models.resnet18(pretrained=True)

        # Modify first conv layer for grayscale input
        self.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

        # Copy pretrained weights (average across RGB channels)
        with torch.no_grad():
            self.conv1.weight = nn.Parameter(
                resnet.conv1.weight.mean(dim=1, keepdim=True)
            )

        # Rest of ResNet
        self.bn1 = resnet.bn1
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4
        self.avgpool = resnet.avgpool

        # Embedding layer
        self.fc = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, embedding_dim)
        )

    def forward_one(self, x):
        """Forward pass for single image"""
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)

        # L2 normalize embedding
        x = F.normalize(x, p=2, dim=1)
        return x

    def forward(self, anchor, positive=None, negative=None):
        """Forward pass for triplet or pair"""
        anchor_out = self.forward_one(anchor)

        if positive is not None and negative is not None:
            # Triplet mode
            positive_out = self.forward_one(positive)
            negative_out = self.forward_one(negative)
            return anchor_out, positive_out, negative_out
        elif positive is not None:
            # Pair mode
            positive_out = self.forward_one(positive)
            return anchor_out, positive_out
        else:
            # Single embedding mode
            return anchor_out


class TamperDetectionCNN(nn.Module):
    """
    Lightweight CNN for detecting digital tampering in signatures.
    Detects copy-paste, noise inconsistencies, and edge artifacts.
    """
    def __init__(self):
        super(TamperDetectionCNN, self).__init__()

        # Feature extraction
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            # Block 4
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((4, 4))
        )

        # Classifier
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, 2)  # Binary: clean vs tampered
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


# Initialize models
siamese_model = SiameseNetwork(embedding_dim=128).to(device)
tamper_model = TamperDetectionCNN().to(device)

print("✅ Models initialized!")
print(f"   Siamese Network parameters: {sum(p.numel() for p in siamese_model.parameters()):,}")
print(f"   Tamper CNN parameters: {sum(p.numel() for p in tamper_model.parameters()):,}")

✅ Models initialized!
   Siamese Network parameters: 11,334,464
   Tamper CNN parameters: 2,746,594


## 📊 Cell 4: Define Datasets and Data Loaders

In [9]:
class SVC2004Dataset(Dataset):
    """
    Dataset for SVC2004 signature verification.

    SVC2004 labeling:
    - S1-S20: Genuine signatures
    - S21-S40: Skilled forgeries
    """
    def __init__(self, root_dirs, transform=None, triplet=True):
        self.transform = transform
        self.triplet = triplet
        self.samples = []  # List of (user_id, sample_id, image_path, is_genuine)
        self.user_samples = {}  # user_id -> {'genuine': [], 'forged': []}

        # Load all images from directories
        for root_dir in root_dirs:
            if not os.path.exists(root_dir):
                continue

            # Walk through all subdirectories
            for dirpath, dirnames, filenames in os.walk(root_dir):
                for filename in filenames:
                    if not filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                        continue

                    filepath = os.path.join(dirpath, filename)

                    # Parse filename to get user and sample IDs
                    # Expected format: U{user}S{sample}.png or similar
                    name = os.path.splitext(filename)[0].upper()

                    # Try to extract user and sample IDs
                    user_id = None
                    sample_id = None

                    if 'U' in name and 'S' in name:
                        try:
                            parts = name.replace('U', '').split('S')
                            user_id = int(parts[0])
                            sample_id = int(parts[1].split('_')[0].split('-')[0])
                        except:
                            pass

                    if user_id is None:
                        # Try alternate naming: user folder structure
                        try:
                            parent = os.path.basename(dirpath)
                            if parent.startswith('U') or parent.startswith('u'):
                                user_id = int(parent[1:])
                                sample_id = int(''.join(filter(str.isdigit, filename.split('.')[0])) or 1)
                        except:
                            continue

                    if user_id is None:
                        continue

                    # S1-S20 = genuine, S21-S40 = forged
                    is_genuine = sample_id <= 20

                    self.samples.append((user_id, sample_id, filepath, is_genuine))

                    if user_id not in self.user_samples:
                        self.user_samples[user_id] = {'genuine': [], 'forged': []}

                    if is_genuine:
                        self.user_samples[user_id]['genuine'].append(filepath)
                    else:
                        self.user_samples[user_id]['forged'].append(filepath)

        # Filter users with at least 2 genuine and 1 forged samples
        valid_users = [uid for uid, samples in self.user_samples.items()
                      if len(samples['genuine']) >= 2 and len(samples['forged']) >= 1]

        self.valid_users = valid_users
        print(f"   Loaded {len(self.samples)} samples from {len(self.valid_users)} valid users")

    def __len__(self):
        return len(self.valid_users) * 20  # Generate multiple triplets per user

    def __getitem__(self, idx):
        # Select random user
        user_id = self.valid_users[idx % len(self.valid_users)]
        user_data = self.user_samples[user_id]

        # Get anchor and positive (both genuine)
        genuine_samples = user_data['genuine']
        indices = np.random.choice(len(genuine_samples), size=2, replace=len(genuine_samples)<2)
        anchor_path = genuine_samples[indices[0]]
        positive_path = genuine_samples[indices[1]]

        # Get negative (forged signature of same user)
        forged_samples = user_data['forged']
        negative_path = np.random.choice(forged_samples)

        # Load images
        anchor = self._load_image(anchor_path)
        positive = self._load_image(positive_path)
        negative = self._load_image(negative_path)

        if self.transform:
            anchor = self.transform(anchor)
            positive = self.transform(positive)
            negative = self.transform(negative)

        return anchor, positive, negative, user_id

    def _load_image(self, path):
        img = Image.open(path).convert('L')  # Grayscale
        return img


class TamperDataset(Dataset):
    """
    Dataset for tamper detection.
    Clean images (label=0) vs Tampered images (label=1)
    """
    def __init__(self, clean_dir, tampered_dir, transform=None):
        self.transform = transform
        self.samples = []

        # Load clean images
        if os.path.exists(clean_dir):
            for root, _, files in os.walk(clean_dir):
                for f in files:
                    if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                        self.samples.append((os.path.join(root, f), 0))

        # Load tampered images
        if os.path.exists(tampered_dir):
            for root, _, files in os.walk(tampered_dir):
                for f in files:
                    if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                        self.samples.append((os.path.join(root, f), 1))

        print(f"   Loaded {len(self.samples)} tamper samples")
        clean_count = sum(1 for _, l in self.samples if l == 0)
        print(f"   Clean: {clean_count}, Tampered: {len(self.samples) - clean_count}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('L')

        if self.transform:
            img = self.transform(img)

        return img, label


# Define transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(5),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

# Create datasets
print("\n" + "="*50)
print("📊 CREATING DATASETS")
print("="*50)

print("\n🔹 SVC2004 Signature Dataset:")
svc_dataset = SVC2004Dataset(
    root_dirs=['datasets/svc2004/task1', 'datasets/svc2004/task2'],
    transform=train_transform,
    triplet=True
)

print("\n🔹 Tamper Detection Dataset:")
tamper_dataset = TamperDataset(
    clean_dir='datasets/tamper/clean',
    tampered_dir='datasets/tamper/tampered',
    transform=train_transform
)

# Split datasets
if len(svc_dataset) > 0:
    train_size = int(0.8 * len(svc_dataset))
    val_size = len(svc_dataset) - train_size
    svc_train, svc_val = torch.utils.data.random_split(svc_dataset, [train_size, val_size])

    svc_train_loader = DataLoader(svc_train, batch_size=32, shuffle=True, num_workers=2)
    svc_val_loader = DataLoader(svc_val, batch_size=32, shuffle=False, num_workers=2)
    print(f"\n✅ SVC2004: Train={train_size}, Val={val_size}")

if len(tamper_dataset) > 0:
    train_size = int(0.8 * len(tamper_dataset))
    val_size = len(tamper_dataset) - train_size
    tamper_train, tamper_val = torch.utils.data.random_split(tamper_dataset, [train_size, val_size])

    tamper_train_loader = DataLoader(tamper_train, batch_size=32, shuffle=True, num_workers=2)
    tamper_val_loader = DataLoader(tamper_val, batch_size=32, shuffle=False, num_workers=2)
    print(f"✅ Tamper: Train={train_size}, Val={val_size}")


📊 CREATING DATASETS

🔹 SVC2004 Signature Dataset:
   Loaded 3200 samples from 40 valid users

🔹 Tamper Detection Dataset:
   Loaded 1550 tamper samples
   Clean: 0, Tampered: 1550

✅ SVC2004: Train=640, Val=160
✅ Tamper: Train=1240, Val=310


## 🏋️ Cell 5: Train Siamese Network for Skilled Forgery Detection

In [ ]:
class TripletLoss(nn.Module):
    """Triplet loss with hard margin"""
    def __init__(self, margin=1.0):
        super(TripletLoss, self).__init__()
        self.margin = margin

    def forward(self, anchor, positive, negative):
        pos_dist = F.pairwise_distance(anchor, positive, p=2)
        neg_dist = F.pairwise_distance(anchor, negative, p=2)
        loss = F.relu(pos_dist - neg_dist + self.margin)
        return loss.mean()


def train_siamese(model, train_loader, val_loader, epochs=50, lr=0.0001):
    """Train Siamese network with triplet loss"""

    criterion = TripletLoss(margin=1.0)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

    history = {'train_loss': [], 'val_loss': [], 'accuracy': []}
    best_loss = float('inf')

    print("\n" + "="*60)
    print("🏋️ TRAINING SIAMESE NETWORK (Skilled Forgery Detection)")
    print("="*60)

    for epoch in range(epochs):
        # Training
        model.train()
        train_loss = 0.0

        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')
        for anchor, positive, negative, _ in pbar:
            anchor = anchor.to(device)
            positive = positive.to(device)
            negative = negative.to(device)

            optimizer.zero_grad()

            anchor_out, positive_out, negative_out = model(anchor, positive, negative)
            loss = criterion(anchor_out, positive_out, negative_out)

            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

        avg_train_loss = train_loss / len(train_loader)

        # Validation
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for anchor, positive, negative, _ in val_loader:
                anchor = anchor.to(device)
                positive = positive.to(device)
                negative = negative.to(device)

                anchor_out, positive_out, negative_out = model(anchor, positive, negative)
                loss = criterion(anchor_out, positive_out, negative_out)
                val_loss += loss.item()

                # Calculate accuracy (positive should be closer than negative)
                pos_dist = F.pairwise_distance(anchor_out, positive_out)
                neg_dist = F.pairwise_distance(anchor_out, negative_out)
                correct += (pos_dist < neg_dist).sum().item()
                total += anchor.size(0)

        avg_val_loss = val_loss / len(val_loader)
        accuracy = correct / total * 100

        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['accuracy'].append(accuracy)

        scheduler.step(avg_val_loss)

        print(f"   Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Accuracy: {accuracy:.2f}%")

        # Save best model
        if avg_val_loss < best_loss:
            best_loss = avg_val_loss
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': best_loss,
            }, 'saved_models/siamese_best.pth')
            print(f"   💾 Best model saved! (Loss: {best_loss:.4f})")

    return history


# Train if dataset is available
if len(svc_dataset) > 0:
    EPOCHS = 30  # Adjust based on your needs (more = better, but slower)
    siamese_history = train_siamese(siamese_model, svc_train_loader, svc_val_loader, epochs=EPOCHS)
else:
    print("⚠️ No SVC2004 data found. Please upload task1.zip and task2.zip")


🏋️ TRAINING SIAMESE NETWORK (Skilled Forgery Detection)


Epoch 1/30:   5%|▌         | 1/20 [00:34<10:52, 34.33s/it, loss=1.0554]

## 🏋️ Cell 6: Train Tamper Detection CNN

In [ ]:
def train_tamper_detector(model, train_loader, val_loader, epochs=30, lr=0.0001):
    """Train tamper detection CNN"""

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=5, factor=0.5)

    history = {'train_loss': [], 'val_loss': [], 'accuracy': [], 'precision': [], 'recall': [], 'f1': []}
    best_acc = 0.0

    print("\n" + "="*60)
    print("🏋️ TRAINING TAMPER DETECTION CNN")
    print("="*60)

    for epoch in range(epochs):
        # Training
        model.train()
        train_loss = 0.0

        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')
        for images, labels in pbar:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

        avg_train_loss = train_loss / len(train_loader)

        # Validation
        model.eval()
        val_loss = 0.0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

                _, preds = torch.max(outputs, 1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        avg_val_loss = val_loss / len(val_loader)

        # Calculate metrics
        accuracy = (np.array(all_preds) == np.array(all_labels)).mean() * 100
        precision = precision_score(all_labels, all_preds, average='binary', zero_division=0) * 100
        recall = recall_score(all_labels, all_preds, average='binary', zero_division=0) * 100
        f1 = f1_score(all_labels, all_preds, average='binary', zero_division=0) * 100

        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['accuracy'].append(accuracy)
        history['precision'].append(precision)
        history['recall'].append(recall)
        history['f1'].append(f1)

        scheduler.step(accuracy)

        print(f"   Loss: {avg_val_loss:.4f} | Acc: {accuracy:.2f}% | P: {precision:.2f}% | R: {recall:.2f}% | F1: {f1:.2f}%")

        # Save best model
        if accuracy > best_acc:
            best_acc = accuracy
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'accuracy': best_acc,
            }, 'saved_models/tamper_best.pth')
            print(f"   💾 Best model saved! (Accuracy: {best_acc:.2f}%)")

    return history


# Train if dataset is available
if len(tamper_dataset) > 0:
    EPOCHS = 30  # Adjust based on your needs
    tamper_history = train_tamper_detector(tamper_model, tamper_train_loader, tamper_val_loader, epochs=EPOCHS)
else:
    print("⚠️ No tamper data found. Please upload synthetic_tamper.zip")

## 📈 Cell 7: Evaluate Models & Calculate Metrics (FAR, FRR, EER)

In [ ]:
def calculate_verification_metrics(model, dataset, threshold=0.5):
    """
    Calculate FAR, FRR, EER for signature verification.

    FAR = False Acceptance Rate (forgeries accepted as genuine)
    FRR = False Rejection Rate (genuine rejected as forgeries)
    EER = Equal Error Rate (where FAR = FRR)
    """
    model.eval()

    genuine_distances = []  # Distances between genuine pairs
    forged_distances = []   # Distances between genuine and forged

    print("\n📊 Computing verification metrics...")

    with torch.no_grad():
        for user_id in tqdm(dataset.valid_users[:50], desc="Processing users"):  # Sample 50 users
            user_data = dataset.user_samples[user_id]
            genuine_paths = user_data['genuine']
            forged_paths = user_data['forged']

            if len(genuine_paths) < 2 or len(forged_paths) < 1:
                continue

            # Get embeddings for genuine signatures
            genuine_embeddings = []
            for path in genuine_paths[:10]:  # Max 10 genuine per user
                img = Image.open(path).convert('L')
                img = val_transform(img).unsqueeze(0).to(device)
                emb = model.forward_one(img)
                genuine_embeddings.append(emb)

            # Get embeddings for forged signatures
            forged_embeddings = []
            for path in forged_paths[:10]:  # Max 10 forged per user
                img = Image.open(path).convert('L')
                img = val_transform(img).unsqueeze(0).to(device)
                emb = model.forward_one(img)
                forged_embeddings.append(emb)

            # Calculate distances
            # Genuine-Genuine pairs (should be small)
            for i in range(len(genuine_embeddings)):
                for j in range(i+1, len(genuine_embeddings)):
                    dist = F.pairwise_distance(genuine_embeddings[i], genuine_embeddings[j]).item()
                    genuine_distances.append(dist)

            # Genuine-Forged pairs (should be large)
            for g_emb in genuine_embeddings:
                for f_emb in forged_embeddings:
                    dist = F.pairwise_distance(g_emb, f_emb).item()
                    forged_distances.append(dist)

    genuine_distances = np.array(genuine_distances)
    forged_distances = np.array(forged_distances)

    # Calculate FAR and FRR at different thresholds
    thresholds = np.linspace(0, 2, 100)
    fars = []
    frrs = []

    for t in thresholds:
        # FAR: forged signatures with distance < threshold (incorrectly accepted)
        far = (forged_distances < t).mean() if len(forged_distances) > 0 else 0
        # FRR: genuine signatures with distance >= threshold (incorrectly rejected)
        frr = (genuine_distances >= t).mean() if len(genuine_distances) > 0 else 0
        fars.append(far)
        frrs.append(frr)

    fars = np.array(fars)
    frrs = np.array(frrs)

    # Find EER (where FAR ≈ FRR)
    eer_idx = np.argmin(np.abs(fars - frrs))
    eer = (fars[eer_idx] + frrs[eer_idx]) / 2
    eer_threshold = thresholds[eer_idx]

    return {
        'far': fars[eer_idx] * 100,
        'frr': frrs[eer_idx] * 100,
        'eer': eer * 100,
        'eer_threshold': eer_threshold,
        'thresholds': thresholds,
        'fars': fars,
        'frrs': frrs,
        'genuine_distances': genuine_distances,
        'forged_distances': forged_distances
    }


def plot_metrics(siamese_metrics=None, tamper_history=None):
    """Plot all evaluation metrics"""

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    # Plot 1: FAR/FRR curves
    if siamese_metrics:
        ax = axes[0, 0]
        ax.plot(siamese_metrics['thresholds'], siamese_metrics['fars'] * 100, 'r-', label='FAR')
        ax.plot(siamese_metrics['thresholds'], siamese_metrics['frrs'] * 100, 'b-', label='FRR')
        ax.axvline(x=siamese_metrics['eer_threshold'], color='g', linestyle='--', label=f'EER Threshold')
        ax.set_xlabel('Distance Threshold')
        ax.set_ylabel('Error Rate (%)')
        ax.set_title(f"FAR/FRR Curves (EER = {siamese_metrics['eer']:.2f}%)")
        ax.legend()
        ax.grid(True, alpha=0.3)

        # Plot 2: Distance distributions
        ax = axes[0, 1]
        ax.hist(siamese_metrics['genuine_distances'], bins=30, alpha=0.6, label='Genuine Pairs', color='green')
        ax.hist(siamese_metrics['forged_distances'], bins=30, alpha=0.6, label='Forged Pairs', color='red')
        ax.axvline(x=siamese_metrics['eer_threshold'], color='black', linestyle='--', label='Threshold')
        ax.set_xlabel('Distance')
        ax.set_ylabel('Frequency')
        ax.set_title('Distance Distribution')
        ax.legend()
        ax.grid(True, alpha=0.3)

    # Plot 3: Siamese training loss
    if 'siamese_history' in dir() and siamese_history:
        ax = axes[0, 2]
        ax.plot(siamese_history['train_loss'], 'b-', label='Train Loss')
        ax.plot(siamese_history['val_loss'], 'r-', label='Val Loss')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss')
        ax.set_title('Siamese Network Training')
        ax.legend()
        ax.grid(True, alpha=0.3)

    # Plot 4: Tamper detection accuracy
    if tamper_history:
        ax = axes[1, 0]
        ax.plot(tamper_history['accuracy'], 'g-', linewidth=2)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Accuracy (%)')
        ax.set_title(f"Tamper Detection Accuracy (Best: {max(tamper_history['accuracy']):.2f}%)")
        ax.grid(True, alpha=0.3)

        # Plot 5: Precision, Recall, F1
        ax = axes[1, 1]
        ax.plot(tamper_history['precision'], 'b-', label='Precision')
        ax.plot(tamper_history['recall'], 'r-', label='Recall')
        ax.plot(tamper_history['f1'], 'g-', label='F1 Score')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Score (%)')
        ax.set_title('Tamper Detection Metrics')
        ax.legend()
        ax.grid(True, alpha=0.3)

        # Plot 6: Training/Val Loss
        ax = axes[1, 2]
        ax.plot(tamper_history['train_loss'], 'b-', label='Train Loss')
        ax.plot(tamper_history['val_loss'], 'r-', label='Val Loss')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss')
        ax.set_title('Tamper CNN Training')
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('results/training_metrics.png', dpi=150)
    plt.show()
    print("\n📊 Metrics plot saved to results/training_metrics.png")


# Evaluate Siamese model
print("\n" + "="*60)
print("📈 EVALUATION METRICS")
print("="*60)

siamese_metrics = None
if len(svc_dataset) > 0:
    # Load best model
    if os.path.exists('saved_models/siamese_best.pth'):
        checkpoint = torch.load('saved_models/siamese_best.pth')
        siamese_model.load_state_dict(checkpoint['model_state_dict'])

    siamese_metrics = calculate_verification_metrics(siamese_model, svc_dataset)

    print("\n" + "="*40)
    print("🔐 SIAMESE NETWORK (Skilled Forgery Detection)")
    print("="*40)
    print(f"   FAR (False Acceptance Rate): {siamese_metrics['far']:.2f}%")
    print(f"   FRR (False Rejection Rate):  {siamese_metrics['frr']:.2f}%")
    print(f"   EER (Equal Error Rate):      {siamese_metrics['eer']:.2f}%")
    print(f"   Optimal Threshold:           {siamese_metrics['eer_threshold']:.4f}")
    print(f"   Accuracy:                    {100 - siamese_metrics['eer']:.2f}%")

# Print tamper detection metrics
if 'tamper_history' in dir() and tamper_history:
    print("\n" + "="*40)
    print("🔍 TAMPER DETECTION CNN")
    print("="*40)
    print(f"   Best Accuracy:  {max(tamper_history['accuracy']):.2f}%")
    print(f"   Best Precision: {max(tamper_history['precision']):.2f}%")
    print(f"   Best Recall:    {max(tamper_history['recall']):.2f}%")
    print(f"   Best F1 Score:  {max(tamper_history['f1']):.2f}%")

# Plot metrics
try:
    plot_metrics(siamese_metrics, tamper_history if 'tamper_history' in dir() else None)
except:
    print("⚠️ Could not generate plots (missing data)")

In [ ]:
if 'siamese_metrics' not in globals():
    siamese_metrics = None

print("\n" + "="*70)
print("📊 FINAL PROJECT METRICS")
print("="*70)

# ----------------------------
# 🔐 Siamese Verification Metrics
# ----------------------------
if siamese_metrics:
    FAR = siamese_metrics['far']
    FRR = siamese_metrics['frr']
    EER = siamese_metrics['eer']
    VER_ACC = 100 - EER   # Verification Accuracy

    print("\n🔐 Signature Verification (Siamese Network)")
    print("-"*50)
    print(f"Verification Accuracy : {VER_ACC:.2f}%")
    print(f"False Acceptance Rate  : {FAR:.2f}%")
    print(f"False Rejection Rate   : {FRR:.2f}%")
    print(f"Equal Error Rate (EER) : {EER:.2f}%")
else:
    print("\n⚠️ Siamese metrics not available.")

# ----------------------------
# 🔍 Tamper Detection Metrics
# ----------------------------
if 'tamper_history' in dir() and tamper_history:
    best_acc = max(tamper_history['accuracy'])
    best_prec = max(tamper_history['precision'])
    best_rec = max(tamper_history['recall'])
    best_f1 = max(tamper_history['f1'])

    print("\n🔍 Tamper Detection CNN")
    print("-"*50)
    print(f"Accuracy  : {best_acc:.2f}%")
    print(f"Precision : {best_prec:.2f}%")
    print(f"Recall    : {best_rec:.2f}%")
    print(f"F1 Score  : {best_f1:.2f}%")
else:
    print("\n⚠️ Tamper metrics not available.")

print("\n" + "="*70)
print("✅ Metrics Ready for PPT / Report")
print("="*70)

## 💾 Cell 8: Save Final Models for Deployment

In [ ]:
# Save models with metadata
print("\n" + "="*60)
print("💾 SAVING MODELS FOR DEPLOYMENT")
print("="*60)

# Siamese model
siamese_save_path = 'saved_models/siamese_production.pth'
torch.save({
    'model_state_dict': siamese_model.state_dict(),
    'embedding_dim': 128,
    'threshold': siamese_metrics['eer_threshold'] if siamese_metrics else 0.8,
    'metrics': {
        'far': siamese_metrics['far'] if siamese_metrics else None,
        'frr': siamese_metrics['frr'] if siamese_metrics else None,
        'eer': siamese_metrics['eer'] if siamese_metrics else None,
    }
}, siamese_save_path)
print(f"✅ Siamese model saved: {siamese_save_path}")

# Tamper model
tamper_save_path = 'saved_models/tamper_production.pth'
torch.save({
    'model_state_dict': tamper_model.state_dict(),
    'metrics': {
        'accuracy': max(tamper_history['accuracy']) if 'tamper_history' in dir() and tamper_history else None,
        'precision': max(tamper_history['precision']) if 'tamper_history' in dir() and tamper_history else None,
        'recall': max(tamper_history['recall']) if 'tamper_history' in dir() and tamper_history else None,
        'f1': max(tamper_history['f1']) if 'tamper_history' in dir() and tamper_history else None,
    }
}, tamper_save_path)
print(f"✅ Tamper model saved: {tamper_save_path}")

# List saved files
print("\n📁 Saved files:")
for f in os.listdir('saved_models'):
    size = os.path.getsize(f'saved_models/{f}') / (1024*1024)
    print(f"   {f} ({size:.2f} MB)")

print("\n⬇️ Download these files from the file browser to use in production.")

---

# 🌐 Part 2: Flask API Server for Frontend Integration

Run this section AFTER training is complete to serve the models via API.

## 🔑 Cell 9: Setup ngrok Authentication

1. Go to https://ngrok.com and create a free account
2. Get your auth token from https://dashboard.ngrok.com/get-started/your-authtoken
3. Paste it below

In [ ]:
# Enter your ngrok auth token here
NGROK_AUTH_TOKEN = "YOUR_NGROK_TOKEN_HERE"  # <-- REPLACE THIS

from pyngrok import ngrok

# Authenticate ngrok
if NGROK_AUTH_TOKEN != "YOUR_NGROK_TOKEN_HERE":
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("✅ ngrok authenticated!")
else:
    print("⚠️ Please enter your ngrok auth token above!")
    print("   Get it from: https://dashboard.ngrok.com/get-started/your-authtoken")

## 🚀 Cell 10: Start Flask API Server

In [ ]:
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
import base64
import io
import hashlib
import threading
import time

# Create Flask app
app = Flask(__name__)
CORS(app)  # Enable CORS for frontend access

# ============================================================
# TAMPER MODEL RELIABILITY CHECK
# ============================================================
# Check if tamper model was trained with BOTH classes (clean + tampered)
# If trained with only one class (e.g., 0 clean images), it is unreliable
tamper_model_reliable = False

# Load models (use production versions if available)
print("🔄 Loading models...")

# Load Siamese model
if os.path.exists('saved_models/siamese_production.pth'):
    checkpoint = torch.load('saved_models/siamese_production.pth', map_location=device)
    siamese_model.load_state_dict(checkpoint['model_state_dict'])
    siamese_threshold = checkpoint.get('threshold', 0.8)
    print(f"   ✅ Siamese model loaded (threshold: {siamese_threshold:.4f})")
elif os.path.exists('saved_models/siamese_best.pth'):
    checkpoint = torch.load('saved_models/siamese_best.pth', map_location=device)
    siamese_model.load_state_dict(checkpoint['model_state_dict'])
    siamese_threshold = 0.8
    print(f"   ✅ Siamese model loaded (using default threshold)")
else:
    siamese_threshold = 0.8
    print("   ⚠️ Using untrained Siamese model")

# Load Tamper model - check if it was properly trained
if os.path.exists('saved_models/tamper_production.pth'):
    checkpoint = torch.load('saved_models/tamper_production.pth', map_location=device)
    tamper_model.load_state_dict(checkpoint['model_state_dict'])
    # Check if metrics indicate proper training (needs both classes)
    metrics = checkpoint.get('metrics', {})
    if metrics.get('accuracy') and metrics['accuracy'] > 60:
        tamper_model_reliable = True
        print("   ✅ Tamper model loaded (reliable)")
    else:
        print("   ⚠️ Tamper model loaded but may be unreliable (low accuracy or missing metrics)")
elif os.path.exists('saved_models/tamper_best.pth'):
    checkpoint = torch.load('saved_models/tamper_best.pth', map_location=device)
    tamper_model.load_state_dict(checkpoint['model_state_dict'])
    acc = checkpoint.get('accuracy', 0)
    if acc > 60:
        tamper_model_reliable = True
        print(f"   ✅ Tamper model loaded (accuracy: {acc:.2f}%)")
    else:
        print(f"   ⚠️ Tamper model loaded but unreliable (accuracy: {acc:.2f}%)")
else:
    print("   ⚠️ Using untrained Tamper model (DISABLED for verification)")

# Also check dataset counts - if clean_count was 0, tamper model is unreliable
if 'clean_count' in dir() and clean_count == 0:
    tamper_model_reliable = False
    print("   🚫 Tamper CNN DISABLED: No clean images in training data (0 clean vs tampered)")
    print("      → Verification will use Siamese Network only")

print(f"\n   📋 Tamper CNN Status: {'✅ ENABLED' if tamper_model_reliable else '🚫 DISABLED (bypassed)'}")

siamese_model.eval()
tamper_model.eval()


def get_raw_base64(base64_string):
    """Extract raw base64 data (strip data URL prefix)"""
    if ',' in base64_string:
        return base64_string.split(',')[1]
    return base64_string


def preprocess_image(base64_string):
    """Convert base64 image to tensor"""
    raw_b64 = get_raw_base64(base64_string)
    image_data = base64.b64decode(raw_b64)
    image = Image.open(io.BytesIO(image_data)).convert('L')
    tensor = val_transform(image).unsqueeze(0).to(device)
    return tensor


def compute_md5(base64_string):
    """Compute MD5 hash of the raw image bytes"""
    raw_b64 = get_raw_base64(base64_string)
    image_data = base64.b64decode(raw_b64)
    return hashlib.md5(image_data).hexdigest()


@app.route('/health', methods=['GET'])
def health_check():
    return jsonify({
        'status': 'healthy',
        'models': {
            'siamese': True,
            'tamper': tamper_model_reliable
        },
        'device': str(device)
    })


@app.route('/verify', methods=['POST'])
def verify_signature():
    """Main verification endpoint with MD5 check and tamper safety"""
    try:
        data = request.get_json()
        reference_b64 = data.get('reference')
        test_b64 = data.get('test')

        if not reference_b64 or not test_b64:
            return jsonify({'error': 'Missing reference or test image'}), 400

        # ==========================================
        # STEP 0: MD5 Hash Check (identical images)
        # ==========================================
        ref_hash = compute_md5(reference_b64)
        test_hash = compute_md5(test_b64)

        if ref_hash == test_hash:
            # Identical images → definitely genuine
            print(f"   🔗 MD5 match: {ref_hash} - returning GENUINE")
            return jsonify({
                'result': 'genuine',
                'siameseScore': 1.0,
                'tamperScore': 0.0,
                'confidence': 1.0,
                'distance': 0.0,
                'threshold': round(siamese_threshold, 4),
                'md5_match': True,
                'details': {
                    'strokeConsistency': 1.0,
                    'pressurePattern': 1.0,
                    'spatialAlignment': 1.0,
                    'pixelAnomalies': 0.0
                }
            })

        # Preprocess images
        ref_tensor = preprocess_image(reference_b64)
        test_tensor = preprocess_image(test_b64)

        with torch.no_grad():
            # ==========================================
            # STEP 1: Tamper Detection (ONLY if reliable)
            # ==========================================
            tamper_score = 0.0
            is_tampered = False

            if tamper_model_reliable:
                tamper_output = tamper_model(test_tensor)
                tamper_probs = F.softmax(tamper_output, dim=1)
                tamper_score = tamper_probs[0, 1].item()
                is_tampered = tamper_score > 0.5
                print(f"   🔍 Tamper score: {tamper_score:.4f} ({'TAMPERED' if is_tampered else 'CLEAN'})")
            else:
                print("   ⏭️ Tamper CNN bypassed (unreliable model)")

            # ==========================================
            # STEP 2: Siamese Verification (ALWAYS runs)
            # ==========================================
            ref_embedding = siamese_model.forward_one(ref_tensor)
            test_embedding = siamese_model.forward_one(test_tensor)

            # Cosine similarity (more stable than pairwise distance)
            cosine_sim = F.cosine_similarity(ref_embedding, test_embedding).item()
            # Also compute distance for threshold comparison
            distance = F.pairwise_distance(ref_embedding, test_embedding).item()
            siamese_score = max(0, min(1, cosine_sim))  # Clamp to [0, 1]

            print(f"   🧠 Siamese: cosine_sim={cosine_sim:.4f}, distance={distance:.4f}, threshold={siamese_threshold:.4f}")

            # ==========================================
            # STEP 3: Decision Fusion
            # ==========================================
            if is_tampered and tamper_model_reliable:
                result = 'tampered'
                confidence = tamper_score
            elif distance < siamese_threshold:
                result = 'genuine'
                confidence = siamese_score
            else:
                result = 'forged'
                confidence = 1 - siamese_score

            print(f"   ✅ Result: {result.upper()} (confidence: {confidence:.4f})")

            # Calculate detailed metrics from embeddings
            ref_flat = ref_embedding.cpu().numpy().flatten()
            test_flat = test_embedding.cpu().numpy().flatten()

            def safe_corr(a, b):
                """Safe correlation that handles constant arrays"""
                if np.std(a) < 1e-8 or np.std(b) < 1e-8:
                    return 0.0
                return max(0, float(np.corrcoef(a, b)[0, 1]))

            stroke_consistency = safe_corr(ref_flat[:32], test_flat[:32])
            pressure_pattern = safe_corr(ref_flat[32:64], test_flat[32:64])
            spatial_alignment = safe_corr(ref_flat[64:96], test_flat[64:96])
            pixel_anomalies = tamper_score

        return jsonify({
            'result': result,
            'siameseScore': round(siamese_score, 4),
            'tamperScore': round(tamper_score, 4),
            'confidence': round(confidence, 4),
            'distance': round(distance, 4),
            'threshold': round(siamese_threshold, 4),
            'tamper_model_active': tamper_model_reliable,
            'details': {
                'strokeConsistency': round(stroke_consistency, 4),
                'pressurePattern': round(pressure_pattern, 4),
                'spatialAlignment': round(spatial_alignment, 4),
                'pixelAnomalies': round(pixel_anomalies, 4)
            }
        })

    except Exception as e:
        import traceback
        traceback.print_exc()
        return jsonify({'error': str(e)}), 500


# Start ngrok tunnel
print("\n" + "="*60)
print("🚀 STARTING API SERVER")
print("="*60)

ngrok.kill()
public_url = ngrok.connect(5000)
print(f"\n🌐 PUBLIC URL: {public_url}")
print("\n" + "="*60)
print("📋 COPY THIS URL TO YOUR FRONTEND .env.local FILE:")
print("="*60)
print(f"\nVITE_API_URL={public_url}")
print("\n" + "="*60)

def run_flask():
    app.run(host='0.0.0.0', port=5000)

flask_thread = threading.Thread(target=run_flask)
flask_thread.daemon = True
flask_thread.start()

print("\n✅ Server is running! Keep this cell running.")
print(f"   Tamper CNN: {'ENABLED' if tamper_model_reliable else 'DISABLED (Siamese-only mode)'}")
print("\n📝 Endpoints:")
print(f"   GET  {public_url}/health  - Health check")
print(f"   POST {public_url}/verify  - Verify signature")

while True:
    time.sleep(60)
    print(f"⏰ Server running... ({time.strftime('%H:%M:%S')})")

---

# 📖 Quick Reference

## Complete Workflow

1. **Upload datasets** (ZIP files via Colab file browser)
2. **Run Cells 1-4** to load and prepare data
3. **Run Cells 5-6** to train both models
4. **Run Cell 7** to evaluate and see metrics
5. **Run Cell 8** to save models
6. **Run Cells 9-10** to start the API server
7. **Copy the ngrok URL** to your frontend `.env.local`

## Expected Metrics

| Model | Metric | Target |
|-------|--------|--------|
| Siamese | EER | < 10% |
| Siamese | Accuracy | > 90% |
| Tamper | Accuracy | > 95% |
| Tamper | F1 Score | > 90% |

## Troubleshooting

- **Out of memory**: Reduce batch size to 16
- **Slow training**: Reduce epochs or use smaller dataset
- **ngrok error**: Get a free token from ngrok.com
- **Frontend can't connect**: Check that ngrok URL is correct in .env.local